In [ ]:
# | default_exp international
# | export
import re
import unicodedata

import regex
from text_to_num import alpha2digit


class _LatinTextNormalizer:
    """Shared conservative normalization for space-separated Latin-script languages."""

    language: str

    def _normalize_numeric_separators(self, text: str) -> str:
        return text

    def _preprocess(self, text: str) -> str:
        return text

    def __call__(self, text: str) -> str:
        # NFC retains meaningful accents while making canonically equivalent text comparable.
        text = unicodedata.normalize("NFC", text).lower()
        text = re.sub(r"[<\[][^>\]]*[>\]]", "", text)
        text = re.sub(r"\(([^)]+?)\)", "", text)
        text = text.translate(str.maketrans({"\u2019": "'", "`": "'", "\u2010": "-", "\u2013": "-", "\u2014": "-"}))
        text = text.replace("\u200b", "").replace("\ufeff", "")
        text = alpha2digit(self._preprocess(text), self.language, threshold=0)
        text = self._normalize_numeric_separators(text)
        # Punctuation is a word boundary for Whisper-style comparison; accents remain intact.
        text = regex.sub(r"(?<!\d)[\p{P}\p{S}]+|[\p{P}\p{S}]+(?!\d)", " ", text)
        return re.sub(r"\s+", " ", text).strip()


class FrenchTextNormalizer(_LatinTextNormalizer):
    """Normalize French text without removing accents or ligatures."""

    language = "fr"

    def _normalize_numeric_separators(self, text: str) -> str:
        # French commonly groups thousands with a regular, no-break, or narrow no-break space.
        text = re.sub(r"(?<=\d)[ \u00a0\u202f](?=\d{3}(?:\D|$))", "", text)
        return re.sub(r"(?<=\d),(?=\d)", ".", text)


class SpanishTextNormalizer(_LatinTextNormalizer):
    """Normalize Spanish text without removing accents, dieresis, or enye."""

    language = "es"

    def _preprocess(self, text: str) -> str:
        return re.sub("\\bveinti\\u00fan\\b", "veintiuno", text)

    def _normalize_numeric_separators(self, text: str) -> str:
        # In Spanish, dots group thousands and commas separate decimal fractions.
        text = re.sub(r"(?<=\d)\.(?=\d{3}(?:\D|$))", "", text)
        return re.sub(r"(?<=\d),(?=\d)", ".", text)



In [ ]:
french_normalizer = FrenchTextNormalizer()
assert french_normalizer("J\u2019ai pay\u00e9 vingt et un euros pour l\u2019\u00e9t\u00e9.") == "j ai pay\u00e9 21 euros pour l \u00e9t\u00e9"
assert french_normalizer("Le total est 1\u202f234,50 \u20ac.") == "le total est 1234.50"

spanish_normalizer = SpanishTextNormalizer()
assert spanish_normalizer("\u00bfPag\u00f3 veinti\u00fan euros?") == "pag\u00f3 21 euros"
assert spanish_normalizer("El total es 1.234,50 \u20ac.") == "el total es 1234.50"
assert spanish_normalizer("El ni\u00f1o lleg\u00f3.") == "el ni\u00f1o lleg\u00f3"
